#Final Project


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import os
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from transformers import EarlyStoppingCallback, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount('/content/drive')

# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

# Function to process FakeNewsNet data
def process_fakenewsnet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []
    label_map = {"HF": "human_fake", "HR": "human_real", "MF": "gpt_fake", "MR": "gpt_real"}

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)
        source = "gossipcop" if "gossipcop" in category else "politifact"
        label = label_map.get(category.split("_")[-1], "unknown")

        for news_id, news_content in data.items():
            if 'text' in news_content:
                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),
                    "source": source,
                    "label": label,
                })

    return pd.DataFrame(data_list)

# Define dataset paths
fake_news_paths = {category: f"/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/FakeNewsNet/{category}.json"
                    for category in ["gossipcop_HF", "gossipcop_HR", "politifact_HF", "politifact_HR",
                                     "gossipcop_MF", "gossipcop_MR", "politifact_MF", "politifact_MR"]}

# Load and process the data
FakeNewsNet_df = process_fakenewsnet_data(fake_news_paths)

# Function to clean text
def clean_text(text):
    """Clean text by removing URLs, extra spaces and converting to lowercase"""
    # Lowercase the text
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the text cleaning function
FakeNewsNet_df["cleaned_text"] = FakeNewsNet_df["text"].apply(clean_text)

# Filter the relevant columns
FakeNewsNet_df = FakeNewsNet_df[["cleaned_text", "label"]]

# balance the data
human_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_real'].sample(n=4181, random_state=42)
gpt_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_real'].sample(n=4181, random_state=42)
human_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_fake']
gpt_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_fake']

# Split each label dataset into train (80%), validation (10%), and test (10%)
def split_dataset(df, label_name):
    train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
    return train, val, test

# Apply the split function to each label category
human_real_train, human_real_val, human_real_test = split_dataset(human_real_df, "human_real")
human_fake_train, human_fake_val, human_fake_test = split_dataset(human_fake_df, "human_fake")
gpt_real_train, gpt_real_val, gpt_real_test = split_dataset(gpt_real_df, "gpt_real")
gpt_fake_train, gpt_fake_val, gpt_fake_test = split_dataset(gpt_fake_df, "gpt_fake")

# Stack the train, validation, and test sets
train_df = pd.concat([human_real_train, human_fake_train, gpt_fake_train, gpt_real_train]).sample(frac=1, random_state=42)
val_df = pd.concat([human_real_val, human_fake_val, gpt_fake_val, gpt_real_val]).sample(frac=1, random_state=42)
test_df = pd.concat([human_real_test, human_fake_test, gpt_fake_test, gpt_real_test]).sample(frac=1, random_state=42)

# Print label distributions
print("Training set label distribution:")
print(train_df['label'].value_counts())
print("\nValidation set label distribution:")
print(val_df['label'].value_counts())
print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Print dataset sizes
print(f"Training: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

# Base Multitask Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

class FocalLoss(nn.Module):
    """Improved Focal Loss implementation with class balancing and numerical stability."""
    def __init__(self, alpha: Optional[Union[float, List[float]]] = None, gamma: float = 2.0, reduction: str = 'mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

        if isinstance(alpha, (float, int)):
            self.alpha = torch.tensor([alpha, 1 - alpha])
        elif isinstance(alpha, list):
            self.alpha = torch.tensor(alpha)
        self.alpha = self.alpha if self.alpha is None else self.alpha / torch.sum(self.alpha)  # Normalize

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)

        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

class DynamicWeightedLoss(nn.Module):
    """Dynamically weighted loss based on task uncertainty."""
    def __init__(self, num_tasks: int):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, *losses: torch.Tensor) -> torch.Tensor:
        total_loss = 0
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total_loss += precision * loss + self.log_vars[i]
        return total_loss

class EnhancedMTLDataset(Dataset):
    """Optimized dataset class with memory mapping and caching."""
    def __init__(self, encodings: Dict[str, torch.Tensor], labels: List[int]):
        self.encodings = encodings
        self.labels = labels

        # Pre-load data to pinned memory if on GPU for faster transfer
        if torch.cuda.is_available():
            for key in self.encodings:
                self.encodings[key] = self.encodings[key].pin_memory()
            self.labels = torch.tensor(self.labels).pin_memory()

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self) -> int:
        return len(self.labels)

class EnhancedMTLModel(nn.Module):
    """Enhanced multitask model with improved architecture and training features."""
    def __init__(self, model_name: str, dropout: float = 0.1,
                 focal_loss_alpha: Optional[Union[float, List[float]]] = None,
                 focal_loss_gamma: float = 2.0,
                 use_weighted_loss: bool = True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # Enhanced shared layers with residual connections
        self.shared_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout)
        )

        # Task-specific heads with separate dropout
        self.source_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
        )
        self.veracity_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
        )

        # Loss functions
        self.focal_loss_fn = FocalLoss(alpha=focal_loss_alpha, gamma=focal_loss_gamma)
        self.use_weighted_loss = use_weighted_loss
        if use_weighted_loss:
            self.task_weighting = DynamicWeightedLoss(num_tasks=2)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize weights for linear layers."""
        for module in [self.shared_layer, self.source_head, self.veracity_head]:
            if isinstance(module, nn.Sequential):
                for layer in module:
                    if isinstance(layer, nn.Linear):
                        nn.init.xavier_uniform_(layer.weight)
                        nn.init.zeros_(layer.bias)

    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                labels: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Use mean pooling instead of just the [CLS] token
        hidden_states = outputs.last_hidden_state
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        pooled_output = torch.sum(hidden_states * attention_mask_expanded, 1) / torch.clamp(attention_mask_expanded.sum(1), min=1e-9)

        shared_features = self.shared_layer(pooled_output)

        source_logits = self.source_head(shared_features)
        veracity_logits = self.veracity_head(shared_features)

        loss = None
        if labels is not None:
            source_labels = (labels // 2).long()
            veracity_labels = (labels % 2).long()

            source_loss = self.focal_loss_fn(source_logits, source_labels)
            veracity_loss = self.focal_loss_fn(veracity_logits, veracity_labels)

            if self.use_weighted_loss:
                loss = self.task_weighting(source_loss, veracity_loss)
            else:
                loss = 0.5 * source_loss + 0.5 * veracity_loss

        return {
            "loss": loss,
            "source_logits": source_logits,
            "veracity_logits": veracity_logits,
            "hidden_states": pooled_output  # For potential later use
        }

class EnhancedMTLTrainerWrapper:
    """Enhanced trainer wrapper with more training options and better evaluation."""
    def __init__(self, model_name: str, output_dir: str,
                 use_fp16: bool = True,
                 gradient_accumulation_steps: int = 1,
                 early_stopping_patience: int = 3):
        self.model_name = model_name
        self.output_dir = output_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.use_fp16 = use_fp16
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.early_stopping_patience = early_stopping_patience

        # Label mappings
        self.label_map = {
            'human_real': 0,
            'human_fake': 1,
            'gpt_real': 2,
            'gpt_fake': 3
        }
        self.reverse_label_map = {v: k for k, v in self.label_map.items()}

        # Initialize in prepare_datasets
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.class_weights = None

    def _encode_labels(self, label_str: List[str]) -> List[int]:
        return [self.label_map[l] for l in label_str]

    def _decode_labels(self, label_num: Union[int, List[int], np.ndarray]) -> Union[str, List[str]]:
        if isinstance(label_num, (list, np.ndarray)):
            return [self.reverse_label_map[int(l)] for l in label_num]
        return self.reverse_label_map[int(label_num)]

    def _prepare_dataset(self, df: pd.DataFrame) -> EnhancedMTLDataset:
        texts = df['cleaned_text'].tolist()
        labels = self._encode_labels(df['label'].tolist())

        encodings = self.tokenizer(
            texts,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return EnhancedMTLDataset(encodings, labels)

    def prepare_datasets(self, train_df: pd.DataFrame,
                        val_df: pd.DataFrame,
                        test_df: pd.DataFrame,
                        compute_class_weights: bool = True):
        self.train_dataset = self._prepare_dataset(train_df)
        self.val_dataset = self._prepare_dataset(val_df)
        self.test_dataset = self._prepare_dataset(test_df)

        if compute_class_weights:
            labels = self.train_dataset.labels
            class_counts = torch.bincount(torch.tensor(labels))
            self.class_weights = (1.0 / class_counts) * len(labels) / len(class_counts)
            self.class_weights = self.class_weights.to(self.device)

    def train(self, epochs: int = 3, batch_size: int = 16, lr: float = 2e-5,
              warmup_ratio: float = 0.1, weight_decay: float = 0.01):
        os.makedirs(self.output_dir, exist_ok=True)

        # Initialize model with class weights if available
        focal_alpha = None
        if self.class_weights is not None:
            focal_alpha = self.class_weights.cpu()

        self.model = EnhancedMTLModel(
            self.model_name,
            focal_loss_alpha=focal_alpha.to(self.device) if focal_alpha is not None else None,
            use_weighted_loss=True
        ).to(self.device)

        # Create dataloaders with persistent workers
        train_loader = DataLoader(
            self.train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True
        )
        val_loader = DataLoader(
            self.val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True
        )

        # Optimizer with layer-wise learning rate decay
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped_parameters = [
            {
                'params': [p for n, p in self.model.named_parameters()
                          if not any(nd in n for nd in no_decay)],
                'weight_decay': weight_decay,
            },
            {
                'params': [p for n, p in self.model.named_parameters()
                          if any(nd in n for nd in no_decay)],
                'weight_decay': 0.0,
            }
        ]
        optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

        # Learning rate scheduler
        total_steps = len(train_loader) * epochs
        warmup_steps = int(total_steps * warmup_ratio)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=total_steps,
            pct_start=warmup_ratio,
            anneal_strategy='linear'
        )

        # Training loop
        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0

            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
            for step, batch in enumerate(progress_bar):
                batch = {k: v.to(self.device) for k, v in batch.items()}

                outputs = self.model(**batch)
                loss = outputs['loss']

                if self.gradient_accumulation_steps > 1:
                    loss = loss / self.gradient_accumulation_steps

                loss.backward()

                if (step + 1) % self.gradient_accumulation_steps == 0:
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                total_loss += loss.item()
                progress_bar.set_postfix({'loss': total_loss / (step + 1)})

            # Validation
            val_metrics = self._evaluate(val_loader)
            val_loss = val_metrics['eval_loss']

            print(f"Epoch {epoch + 1} - Train Loss: {total_loss / len(train_loader):.4f}")
            print(f"Validation Loss: {val_loss:.4f}")
            print(f"Validation Accuracy: {val_metrics['eval_combined_acc']:.4f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(self.model.state_dict(), f"{self.output_dir}/best_model.pt")
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    print(f"Early stopping after {epoch + 1} epochs")
                    break

        # Load best model
        self.model.load_state_dict(torch.load(f"{self.output_dir}/best_model.pt"))

    def _evaluate(self, dataloader: DataLoader) -> Dict[str, float]:
        self.model.eval()
        total_loss = 0
        all_source_preds = []
        all_veracity_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in dataloader:
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)

                total_loss += outputs['loss'].item()

                source_preds = torch.argmax(outputs['source_logits'], dim=1).cpu().numpy()
                veracity_preds = torch.argmax(outputs['veracity_logits'], dim=1).cpu().numpy()
                labels = batch['labels'].cpu().numpy()

                all_source_preds.extend(source_preds)
                all_veracity_preds.extend(veracity_preds)
                all_labels.extend(labels)

        combined_preds = 2 * np.array(all_source_preds) + np.array(all_veracity_preds)
        combined_labels = np.array(all_labels)

        metrics = {
            'eval_loss': total_loss / len(dataloader),
            'eval_combined_acc': accuracy_score(combined_labels, combined_preds),
            'source_acc': accuracy_score(combined_labels // 2, all_source_preds),
            'veracity_acc': accuracy_score(combined_labels % 2, all_veracity_preds)
        }

        return metrics

    def save(self):
        """Save model, tokenizer, and configuration."""
        self.model.encoder.save_pretrained(self.output_dir)
        self.tokenizer.save_pretrained(self.output_dir)

        # Save full model state for easier loading
        torch.save({
            'model_state': self.model.state_dict(),
            'config': {
                'model_name': self.model_name,
                'label_map': self.label_map
            }
        }, f"{self.output_dir}/full_model.pt")

    def load(self):
        """Load model from saved checkpoint."""
        checkpoint = torch.load(f"{self.output_dir}/full_model.pt", map_location=self.device)
        self.model = EnhancedMTLModel(checkpoint['config']['model_name']).to(self.device)
        self.model.load_state_dict(checkpoint['model_state'])
        self.label_map = checkpoint['config']['label_map']
        self.reverse_label_map = {v: k for k, v in self.label_map.items()}
        self.tokenizer = AutoTokenizer.from_pretrained(self.output_dir)

    def predict(self, texts: List[str], batch_size: int = 32) -> List[str]:
        """Batch prediction with progress bar."""
        self.model.eval()
        predictions = []

        for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
            batch_texts = texts[i:i + batch_size]

            encodings = self.tokenizer(
                batch_texts,
                max_length=512,
                padding='max_length',
                truncation=True,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**encodings)
                source_preds = torch.argmax(outputs["source_logits"], dim=1).cpu().numpy()
                veracity_preds = torch.argmax(outputs["veracity_logits"], dim=1).cpu().numpy()

            combined_preds = 2 * source_preds + veracity_preds
            predictions.extend(self._decode_labels(combined_preds))

        return predictions

    def evaluate_on_test(self, test_df: pd.DataFrame) -> Tuple[str, pd.DataFrame]:
        """Enhanced evaluation with more metrics and visualization."""
        true_labels = self._encode_labels(test_df["label"].tolist())
        texts = test_df["cleaned_text"].tolist()
        preds = self.predict(texts)
        preds_encoded = self._encode_labels(preds)

        # Classification report
        report = classification_report(
            true_labels, preds_encoded,
            target_names=["human_real", "human_fake", "gpt_real", "gpt_fake"],
            digits=4,
            output_dict=False
        )

        # Confusion matrix with normalization
        cm = confusion_matrix(true_labels, preds_encoded)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

        cm_df = pd.DataFrame(
            cm_normalized,
            index=["human_real", "human_fake", "gpt_real", "gpt_fake"],
            columns=["human_real", "human_fake", "gpt_real", "gpt_fake"]
        )

        # Enhanced visualization
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_df, annot=True, fmt=".2f", cmap="Blues",
                    cbar=True, vmin=0, vmax=1)
        plt.title("Normalized Confusion Matrix")
        plt.ylabel('True Labels')
        plt.xlabel('Predicted Labels')
        plt.tight_layout()
        plt.show()

        # Additional metrics
        source_true = np.array(true_labels) // 2
        source_pred = np.array(preds_encoded) // 2
        veracity_true = np.array(true_labels) % 2
        veracity_pred = np.array(preds_encoded) % 2

        source_report = classification_report(
            source_true, source_pred,
            target_names=["human", "gpt"],
            digits=4,
            output_dict=False
        )

        veracity_report = classification_report(
            veracity_true, veracity_pred,
            target_names=["real", "fake"],
            digits=4,
            output_dict=False
        )

        print("Source Classification Report:")
        print(source_report)
        print("\nVeracity Classification Report:")
        print(veracity_report)

        return report, cm_df

In [ ]:
# ========== ModernBERT-base ==========
model_path = "/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/Multitask/ModernBERT-base"
trainer = EnhancedMTLTrainerWrapper(
    model_name="answerdotai/ModernBERT-base",
    output_dir="./model_output",
    use_fp16=True,
    early_stopping_patience=2
)

trainer.prepare_datasets(train_df, val_df, test_df, compute_class_weights=True)
trainer.train(epochs=2, batch_size=16, lr=2e-5)
trainer.save()


In [ ]:
test_report, test_cm = trainer.evaluate_on_test(test_df)
print("Classification Report:")
print(test_report)
print("\nConfusion Matrix:")
print(test_cm)
